# Lab | Web Scraping

Welcome to the "Books to Scrape" Web Scraping Adventure Lab!

**Objective**

In this lab, we will embark on a mission to unearth valuable insights from the data available on Books to Scrape, an online platform showcasing a wide variety of books. As data analyst, you have been tasked with scraping a specific subset of book data from Books to Scrape to assist publishing companies in understanding the landscape of highly-rated books across different genres. Your insights will help shape future book marketing strategies and publishing decisions.

**Background**

In a world where data has become the new currency, businesses are leveraging big data to make informed decisions that drive success and profitability. The publishing industry, much like others, utilizes data analytics to understand market trends, reader preferences, and the performance of books based on factors such as genre, author, and ratings. Books to Scrape serves as a rich source of such data, offering detailed information about a diverse range of books, making it an ideal platform for extracting insights to aid in informed decision-making within the literary world.

**Task**

Your task is to create a Python script using BeautifulSoup and pandas to scrape Books to Scrape book data, focusing on book ratings and genres. The script should be able to filter books with ratings above a certain threshold and in specific genres. Additionally, the script should structure the scraped data in a tabular format using pandas for further analysis.

**Expected Outcome**

A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`. The function should scrape book data from the "Books to Scrape" website and return a `pandas` DataFrame with the following columns:

**Expected Outcome**

- A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`.
- The function should return a DataFrame with the following columns:
  - **UPC**: The Universal Product Code (UPC) of the book.
  - **Title**: The title of the book.
  - **Price (£)**: The price of the book in pounds.
  - **Rating**: The rating of the book (1-5 stars).
  - **Genre**: The genre of the book.
  - **Availability**: Whether the book is in stock or not.
  - **Description**: A brief description or product description of the book (if available).
  
You will execute this script to scrape data for books with a minimum rating of `4.0 and above` and a maximum price of `£20`. 

Remember to experiment with different ratings and prices to ensure your code is versatile and can handle various searches effectively!

**Resources**

- [Beautiful Soup Documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [Pandas Documentation](https://pandas.pydata.org/pandas-docs/stable/index.html)
- [Books to Scrape](https://books.toscrape.com/)


**Hint**

Your first mission is to familiarize yourself with the **Books to Scrape** website. Navigate to [Books to Scrape](http://books.toscrape.com/) and explore the available books to understand their layout and structure. 

Next, think about how you can set parameters for your data extraction:

- **Minimum Rating**: Focus on books with a rating of 4.0 and above.
- **Maximum Price**: Filter for books priced up to £20.

After reviewing the site, you can construct a plan for scraping relevant data. Pay attention to the details displayed for each book, including the title, price, rating, and availability. This will help you identify the correct HTML elements to target with your scraping script.

Make sure to build your scraping URL and logic based on the patterns you observe in the HTML structure of the book listings!


---

**Best of luck! Immerse yourself in the world of books, and may the data be with you!**

**Important Note**:

In the fast-changing online world, websites often update and change their structures. When you try this lab, the **Books to Scrape** website might differ from what you expect.

If you encounter issues due to these changes, like new rules or obstacles preventing data extraction, don’t worry! Get creative.

You can choose another website that interests you and is suitable for scraping data. Options like Wikipedia, The New York Times, or even library databases are great alternatives. The main goal remains the same: extract useful data and enhance your web scraping skills while exploring a source of information you enjoy. This is your opportunity to practice and adapt to different web environments!

In [ ]:
!pip install beautifulsoup4 requests pandas

import re
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd

# The site shows the rating as a word in the CSS class (e.g. "star-rating Three")
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

In [ ]:
def get_soup(url):
    """Download a page and return it as a BeautifulSoup object (None if the request fails)."""
    response = requests.get(url, timeout=10)
    if response.status_code != 200:
        return None
    # The site does not declare its encoding, so requests would guess ISO-8859-1
    # and turn "£" into "Â£". Setting UTF-8 avoids that (and broken quotes in descriptions).
    response.encoding = "utf-8"
    return BeautifulSoup(response.text, "html.parser")


def scrape_books(min_rating, max_price):
    """
    Scrapes Books to Scrape website and returns
    a DataFrame of books filtered by minimum rating
    and maximum price.

    Parameters:
        min_rating : int/float - minimum star rating (1-5)
        max_price  : float     - maximum price in pounds

    Returns:
        pandas DataFrame with book details
    """
    base_url = "https://books.toscrape.com/catalogue/"
    url = base_url + "page-1.html"
    books_data = []
    page = 1

    print(f"Scraping books with rating >= {min_rating} and price <= £{max_price}...")

    while url:
        print(f"Scraping page {page}...")
        soup = get_soup(url)
        if soup is None:
            print(f"Failed to fetch page {page}")
            break

        book_list = soup.find_all("article", class_="product_pod")

        for book in book_list:
            # Get rating (the second CSS class of the star-rating paragraph)
            rating_word = book.find("p", class_="star-rating")["class"][1]
            rating = rating_map.get(rating_word, 0)

            # Get price
            price_text = book.find("p", class_="price_color").text.strip()
            price = float(re.sub(r"[^\d.]", "", price_text))

            # Apply filters early to avoid unnecessary requests
            if rating < min_rating or price > max_price:
                continue

            # Get title
            title = book.find("h3").find("a")["title"]

            # Get book detail page URL (relative to the current listing page)
            book_url = urljoin(url, book.find("h3").find("a")["href"])

            # Get detailed info from book detail page
            book_soup = get_soup(book_url)
            if book_soup is None:
                print(f"Skipping '{title}': could not fetch {book_url}")
                continue

            # Get UPC (the product information table has one row per field)
            table = book_soup.find("table", class_="table-striped")
            product_info = {row.find("th").text.strip(): row.find("td").text.strip()
                            for row in table.find_all("tr")}
            upc = product_info["UPC"]

            # Get genre (third item of the breadcrumb: Home > Books > Genre > Title)
            breadcrumb = book_soup.find("ul", class_="breadcrumb")
            genre = breadcrumb.find_all("li")[2].text.strip()

            # Get availability
            availability = " ".join(
                book_soup.find("p", class_="instock availability").text.split()
            )

            # Get description
            description_tag = book_soup.find("div", id="product_description")
            description_p = description_tag.find_next_sibling("p") if description_tag else None
            description = description_p.text.strip() if description_p else "No description available"

            # Append book data
            books_data.append({
                "UPC"          : upc,
                "Title"        : title,
                "Price (£)"    : price,
                "Rating"       : rating,
                "Genre"        : genre,
                "Availability" : availability,
                "Description"  : description
            })

        # Check for next page
        next_button = soup.find("li", class_="next")
        if next_button:
            url = urljoin(url, next_button.find("a")["href"])
            page += 1
        else:
            url = None

    # Create DataFrame (the columns exist even if no book matches)
    df = pd.DataFrame(
        books_data,
        columns=["UPC", "Title", "Price (£)", "Rating", "Genre", "Availability", "Description"],
    )

    print(f"\nScraping complete!")
    print(f"Total books found : {len(df)}")

    return df

In [ ]:
# Run the scraper
df_books = scrape_books(min_rating=4, max_price=20)

# Display results
print("SCRAPING RESULTS")
print(f"\nTotal Books Scraped  : {len(df_books)}")
print(f"Columns              : {df_books.columns.tolist()}")

print("\nSample Data:")
print(df_books.head())

print("\nBooks by Genre:")
print(df_books["Genre"].value_counts())

print("\nRating Distribution:")
print(df_books["Rating"].value_counts().sort_index())

print("\nPrice Statistics:")
print(df_books["Price (£)"].describe().round(2))

# Save to CSV
df_books.to_csv("books_scraped.csv", index=False)
print("\nData saved to books_scraped.csv")